# 02 — Data aggregation and entity matching

**Cilj:** dobiti jednu vrstu po igraču, povezati performance podatke sa Transfermarkt valuacijama i napraviti auditabilan `final_df`.

Redosled matchinga je: **exact → normalized → fuzzy → ambiguous exact**. Prihvataju se samo kandidati koji prolaze definisane name, club i age pragove.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from src.data_processing import build_aggregated_dataset
from src.matching import build_final_dataset, prepare_market_values

aggregated_df, outfield_selected = build_aggregated_dataset(
    RAW_DIR / "players_data-2024_2025.csv"
)
print(f"Selected rows: {len(outfield_selected):,}")
print(f"Aggregated identities: {len(aggregated_df):,}")
aggregated_df.head()

## Identitet i agregacija

`identity_key` polazi od imena i uzrasta. Poznata kolizija dve osobe pod imenom Vitinha (obe 24) eksplicitno se razdvaja klubom u `src/data_processing.py`. `MainSquad`, pozicija i liga dolaze iz reda sa najviše minuta, dok se counting statistike sabiraju. `Cmp%`, `Succ%` i `Won%` ponovo se računaju iz ukupnih brojilaca i imenilaca.

In [ ]:
assert aggregated_df["identity_key"].is_unique
assert len(aggregated_df) == outfield_selected["identity_key"].nunique()

display(
    outfield_selected.loc[outfield_selected["Player"].eq("Vitinha"),
                          ["identity_key", "Player", "Age", "Squad", "Comp", "Min"]]
)
display(aggregated_df[["Min", "Cmp%", "Succ%", "Won%"]].describe())

## Transfermarkt snapshot i višestepeni matching

In [ ]:
market_values, valuations = prepare_market_values(
    RAW_DIR / "players.csv",
    RAW_DIR / "player_valuations.csv",
)
final_df, player_matches, unresolved = build_final_dataset(aggregated_df, market_values)

match_summary = player_matches["match_method"].value_counts().rename("players").to_frame()
match_summary.loc["TOTAL"] = len(player_matches)
display(match_summary)
print(f"Match rate: {len(player_matches) / len(aggregated_df):.1%}")

## Conservation checks

Ove provere sprečavaju tiho dupliranje ili nestanak identiteta tokom join operacija.

In [ ]:
assert len(player_matches) + len(unresolved) == len(aggregated_df)
assert player_matches["identity_key"].is_unique
assert player_matches["player_id"].is_unique
assert final_df["identity_key"].is_unique
assert final_df["player_id"].is_unique
assert final_df["market_value_in_eur"].notna().all()

print(f"Aggregated: {len(aggregated_df):,}")
print(f"Matched:    {len(player_matches):,}")
print(f"Unresolved: {len(unresolved):,}")

## Snimanje procesiranih podataka

`matches.csv` i `unresolved_players.csv` ostaju kao audit trag; `final_players.csv` je ulaz za feature engineering.

In [ ]:
final_df.to_csv(PROCESSED_DIR / "final_players.csv", index=False)
player_matches.to_csv(PROCESSED_DIR / "matches.csv", index=False)
unresolved.to_csv(PROCESSED_DIR / "unresolved_players.csv", index=False)

display(final_df.head())
print(f"Saved {len(final_df):,} matched players to {PROCESSED_DIR}")